In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import sys
import sklearn
from sklearn import metrics
import os
import pandas as pd
import time
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA,KernelPCA
import logging
logging.disable(30)
from collections import  Counter
import math
import random
from fitter import Fitter

In [2]:
data0  = pd.read_csv(r"F:\数据集样本划分\data_test_sensorA_3_600.csv").values
label0 =pd.read_csv(r"F:\数据集样本划分\label_test_sensorA_3_600.csv").values
data1  = pd.read_csv(r"F:\数据集样本划分\data_test_sensorA_3_1200.csv").values
label1 =pd.read_csv(r"F:\数据集样本划分\label_test_sensorA_3_1200.csv").values
data2  = pd.read_csv(r"F:\数据集样本划分\data_test_sensorA_3_2400.csv").values
label2 =pd.read_csv(r"F:\数据集样本划分\label_test_sensorA_3_2400.csv").values

source_data=data0
source_label=label0
target_data=data2
target_label=label2

source_data.shape,source_label.shape,target_data.shape,target_label.shape

((7000, 3072), (7000, 7), (7000, 3072), (7000, 7))

In [3]:
source_data=(source_data-source_data.min(axis=1).reshape((len(source_data),1)))/(source_data.max(axis=1).reshape((len(source_data),1))-source_data.min(axis=1).reshape((len(source_data),1)))
target_data=(target_data-target_data.min(axis=1).reshape((len(target_data),1)))/(target_data.max(axis=1).reshape((len(target_data),1))-target_data.min(axis=1).reshape((len(target_data),1)))
source_data=tf.expand_dims(source_data,axis=-1)
target_data=tf.expand_dims(target_data,axis=-1)
source=source_data

train_dataset=tf.data.Dataset.from_tensor_slices((source_data,target_data,source_label,target_label))
train_dataset=train_dataset.shuffle(50000).batch(512,drop_remainder=True)
test_dataset=tf.data.Dataset.from_tensor_slices((source_data,target_data,source_label,target_label))
test_dataset=test_dataset.batch(7000,drop_remainder=True)

In [4]:
input_1=tf.keras.Input(shape=(3072,1),name='source_data')

x=tf.keras.layers.Conv1D(filters=16,kernel_size=64,strides=16,activation='relu',padding='same',name='conv1')(input_1)
x=tf.keras.layers.BatchNormalization(name='bn1')(x)
x=tf.keras.layers.MaxPooling1D(pool_size=2,strides=2,padding='same',name='plool1')(x)

x=tf.keras.layers.Conv1D(filters=32,kernel_size=3,strides=1,activation='relu',padding='same',name='conv2')(x)
x=tf.keras.layers.BatchNormalization(name='bn2')(x)
x=tf.keras.layers.MaxPooling1D(pool_size=2,strides=2,padding='same',name='plool2')(x)

x=tf.keras.layers.Conv1D(filters=64,kernel_size=3,strides=1,activation='relu',padding='same',name='conv3')(x)
x=tf.keras.layers.BatchNormalization(name='bn3')(x)
x=tf.keras.layers.MaxPooling1D(pool_size=2,strides=2,padding='same',name='plool3')(x)

x=tf.keras.layers.Conv1D(filters=128,kernel_size=3,strides=1,activation='relu',padding='same',name='conv4')(x)
x=tf.keras.layers.BatchNormalization(name='bn4')(x)
x=tf.keras.layers.MaxPooling1D(pool_size=2,strides=2,padding='same',name='plool4')(x)

x=tf.keras.layers.Conv1D(filters=256,kernel_size=3,strides=1,activation='relu',padding='same',name='conv5')(x)
x=tf.keras.layers.BatchNormalization(name='bn5')(x)
x=tf.keras.layers.MaxPooling1D(pool_size=2,strides=2,padding='same',name='plool5')(x)

x=tf.keras.layers.GlobalAveragePooling1D()(x)

y1=tf.keras.layers.Dense(128,activation='relu',name='cl_1')(x)
y2=tf.keras.layers.Dense(7,name='cl_2')(y1)

model=tf.keras.Model(inputs=input_1,outputs=[y1,y2])

In [5]:
pre_optimizer=tf.keras.optimizers.Adam(learning_rate=0.001)
optimizer=tf.keras.optimizers.Adam(learning_rate=0.001)            
loss_func=tf.keras.losses.CategoricalCrossentropy(from_logits=True)

In [6]:
clc_loss=tf.keras.metrics.Mean('clc_loss')          
pre_train_loss=tf.keras.metrics.Mean("pre_train_loss")

train_accuracy=tf.keras.metrics.CategoricalAccuracy('train_accuracy')
pre_train_accuracy=tf.keras.metrics.CategoricalAccuracy('pre_train_accuracy')
test_accuracy=tf.keras.metrics.CategoricalAccuracy('test_accuracy')

train_acc=[]
test_acc=[]
clc_loss_=[]
margin=[]

pre_train_acc=[]
pre_train_loss_=[]

In [7]:
def class_margin(m,a,la):
    if len(la) == 0:
        return a
    else:
        index=tf.argmax(la[0],axis=-1)
    for i in range(len(a)):
        c=a[i]
        part1 = c[:index]
        part2 = c[index+1:]
        val = [ c[index]-m]
    
        if i==0:  
            new_tensor = tf.concat([part1,val,part2], axis=0)
        else :
            tensor=tf.concat([part1,val,part2], axis=0)
            new_tensor=tf.experimental.numpy.vstack([new_tensor,tensor])
    return new_tensor

def adamargin(data):
    pca=PCA(n_components=1,random_state=1000)
    data=pca.fit_transform(data).squeeze()
    
    s1=Fitter(data[0:1000],   distributions=['t','norm'])
    s2=Fitter(data[1000:2000],distributions=['t','norm'])
    s3=Fitter(data[2000:3000],distributions=['t','norm'])
    s4=Fitter(data[3000:4000],distributions=['t','norm'])
    s5=Fitter(data[4000:5000],distributions=['t','norm'])
    s6=Fitter(data[5000:6000],distributions=['t','norm'])
    s7=Fitter(data[6000:7000],distributions=['t','norm'])

    s1.fit()
    s2.fit()
    s3.fit()
    s4.fit()
    s5.fit()
    s6.fit()  
    s7.fit()    
    
    mean=[s1.fitted_param['t'][1],s2.fitted_param['t'][1],s3.fitted_param['t'][1],s4.fitted_param['t'][1]
          ,s5.fitted_param['t'][1],s6.fitted_param['t'][1],s7.fitted_param['t'][1]]
    sta=[s1.fitted_param['t'][2],s2.fitted_param['t'][2],s3.fitted_param['t'][2],s4.fitted_param['t'][2]
         ,s5.fitted_param['t'][2],s6.fitted_param['t'][2],s7.fitted_param['t'][2]]    

    margin_list=[]
    num=len(mean)
    for i in range(num):
        margin=[]
        for j in range(num):
            if j !=i:
                condition=6*(sta[i]+sta[j])-abs(mean[i]-mean[j])
                if condition < 0:
                    margin.append(0.)
                elif condition > 0:
                    margin.append(condition)
        margin_list.append(margin)
    margin_list=np.max(margin_list,axis=1)
    return margin_list

def split(source_out1,source_output1,source_label):
    label_argmax=np.argmax(source_label,axis=-1)
    la,lb,lc,ld,le,lf,lg=[],[],[],[],[],[],[]
    a,b,c,d,e,f,g=[],[],[],[],[],[],[]
    da,db,dc,dd,de,df,dg=[],[],[],[],[],[],[]
    for i in range(source_label.shape[0]):
        if label_argmax[i]==0:
            a.append(source_output1[i])
            la.append(source_label[i])
            da.append(source_out1[i])
        elif label_argmax[i]==1:
            b.append(source_output1[i])
            lb.append(source_label[i])
            db.append(source_out1[i])
        elif label_argmax[i]==2:
            c.append(source_output1[i])
            lc.append(source_label[i])
            dc.append(source_out1[i])
        elif label_argmax[i]==3:
            d.append(source_output1[i])
            ld.append(source_label[i])
            dd.append(source_out1[i])
        elif label_argmax[i]==4:
            e.append(source_output1[i])
            le.append(source_label[i])
            de.append(source_out1[i])
        elif label_argmax[i]==5:
            f.append(source_output1[i])
            lf.append(source_label[i])
            df.append(source_out1[i])            
        elif label_argmax[i]==6:
            g.append(source_output1[i])
            lg.append(source_label[i])
            dg.append(source_out1[i])            
    return a,b,c,d,e,f,g,la,lb,lc,ld,le,lf,lg

def DG_Softmax(mar,source_out1,source_output1,source_label):
    a,b,c,d,e,f,g,la,lb,lc,ld,le,lf,lg=split(source_out1,source_output1,source_label)
    m,n,p,q,k,x,y=mar  
    
    a=tf.convert_to_tensor(class_margin(m,a,la))
    b=tf.convert_to_tensor(class_margin(n,b,lb))
    c=tf.convert_to_tensor(class_margin(p,c,lc))
    d=tf.convert_to_tensor(class_margin(q,d,ld))
    e=tf.convert_to_tensor(class_margin(k,e,le))
    f=tf.convert_to_tensor(class_margin(x,f,lf))    
    g=tf.convert_to_tensor(class_margin(y,g,lg))
    
    data_set=[]
    label_set=[]
    if len(a)!=0:
        data_set.append(a)
        label_set.append(la)        
    if len(b)!=0:
        data_set.append(b)
        label_set.append(lb)
    if len(c)!=0:
        data_set.append(c)
        label_set.append(lc)
    if len(d)!=0:
        data_set.append(d)
        label_set.append(ld)
    if len(e)!=0:
        data_set.append(e)
        label_set.append(le)
    if len(f)!=0:
        data_set.append(f)
        label_set.append(lf)
    if len(g)!=0:
        data_set.append(g)
        label_set.append(lg)
    
    data=tf.experimental.numpy.vstack(data_set)
    label=tf.experimental.numpy.vstack(label_set)
    loss=loss_func(label,data)
    return data,label,loss

In [8]:
def pre_train_step(epoch, source_data,target_data,source_label,target_label):
    with tf.GradientTape() as t:
        out1,output1 = model(source_data,training = True)
        out2,output2 = model(target_data,training = True)                                
        
        clc_loss_step=loss_func(source_label,output1)
        loss_step=clc_loss_step
    
    grads = t.gradient(loss_step, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    clc_loss(clc_loss_step)
    train_accuracy(source_label,output1)
    test_accuracy(target_label,output2)

def pre_train():
    for epoch in range(100):
        for (batch, (source_data,target_data,source_label,target_label)) in enumerate(train_dataset):
            pre_train_step(epoch,source_data,target_data,source_label,target_label)
        train_acc.append(train_accuracy.result())
        clc_loss_.append(clc_loss.result())
        test_acc.append(test_accuracy.result())
        
        print('Epoch{}, clc_loss is {:.5f}, train_accuracy is {:.5f},test_accuracy is {:.5f}'.format(epoch+1,
                                                                                          clc_loss.result(),           
                                                                                          train_accuracy.result(),
                                                                                          test_accuracy.result()))
        
        clc_loss.reset_states()
        train_accuracy.reset_states()
        test_accuracy.reset_states()

In [9]:
def train_step(mar, source_data,target_data,source_label,target_label):
    with tf.GradientTape() as t:
        out1,output1 = model(source_data,training = True)
        out2,output2 = model(target_data,training = True)
        
        data,label,clc_loss_step =DG_Softmax(mar,out1,output1,source_label)
        loss_step=clc_loss_step
    
    grads = t.gradient(loss_step, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    clc_loss(clc_loss_step)
    train_accuracy(label,data)
    test_accuracy(target_label,output2)
    
def train():
    for epoch in range(50):
        if epoch%1==0:
            _,feature=model(source,training = True)
            mar=adamargin(feature)
            ini_margin=mar
        else:
            mar=ini_margin
        
        margin.append(mar)
        print(mar)
        
        for (batch, (source_data,target_data,source_label,target_label)) in enumerate(train_dataset):
            train_step(mar,source_data,target_data,source_label,target_label)
        train_acc.append(train_accuracy.result())
        clc_loss_.append(clc_loss.result())
        test_acc.append(test_accuracy.result())
        
        print('Epoch{}, clc_loss is {:.5f}, train_accuracy is {:.5f},test_accuracy is {:.5f}'.format(epoch+1,
                                                                                          clc_loss.result(),           
                                                                                          train_accuracy.result(),
                                                                                          test_accuracy.result()))
        clc_loss.reset_states()
        train_accuracy.reset_states()
        test_accuracy.reset_states()

In [10]:
pre_train()

Epoch1, clc_loss is 0.50202, train_accuracy is 0.83939,test_accuracy is 0.41211
Epoch2, clc_loss is 0.02464, train_accuracy is 0.99549,test_accuracy is 0.57302
Epoch3, clc_loss is 0.00638, train_accuracy is 0.99940,test_accuracy is 0.59675
Epoch4, clc_loss is 0.00266, train_accuracy is 0.99970,test_accuracy is 0.58894
Epoch5, clc_loss is 0.00162, train_accuracy is 0.99985,test_accuracy is 0.58909
Epoch6, clc_loss is 0.00094, train_accuracy is 1.00000,test_accuracy is 0.59105
Epoch7, clc_loss is 0.00076, train_accuracy is 1.00000,test_accuracy is 0.59961
Epoch8, clc_loss is 0.00068, train_accuracy is 1.00000,test_accuracy is 0.60322
Epoch9, clc_loss is 0.00056, train_accuracy is 1.00000,test_accuracy is 0.59871
Epoch10, clc_loss is 0.00050, train_accuracy is 1.00000,test_accuracy is 0.60562
Epoch11, clc_loss is 0.00040, train_accuracy is 1.00000,test_accuracy is 0.60847
Epoch12, clc_loss is 0.00038, train_accuracy is 1.00000,test_accuracy is 0.60727
Epoch13, clc_loss is 0.00032, train_a

In [11]:
train()

Fitting 2 distributions: 100%|███████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 46.64it/s]


[21.46244965 21.46244965 22.51592779 22.51592779 15.27293318 18.30716182
 13.70964869]
Epoch1, clc_loss is 1.52225, train_accuracy is 0.76938,test_accuracy is 0.57257


Fitting 2 distributions: 100%|███████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 46.46it/s]


[104.595853    45.62334716  78.89687091  47.8033844  104.595853
  47.8033844   40.09560872]
Epoch2, clc_loss is 6.51752, train_accuracy is 0.72611,test_accuracy is 0.65024


Fitting 2 distributions: 100%|███████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 47.57it/s]


...

Fitting 2 distributions: 100%|███████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 45.29it/s]


[67.30639903 72.2422023  84.88615085 81.80284537 84.88615085 67.30639903
 60.87191215]
Epoch10, clc_loss is 0.00526, train_accuracy is 0.99970,test_accuracy is 0.67127


Fitting 2 distributions: 100%|███████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 45.96it/s]


[81.0497155  80.20157806 88.99512833 88.99512833 87.7322191  81.0497155
 66.30244972]


KeyboardInterrupt: 